# **MILESTONE 3**

Nama  : Nur Cahyo Widodo  
Batch : HCK 035 (Pondok Indah)

Notebook ini digunakan untuk melakukan validasi kualitas datamenggunakan Great Expectations. Dataset Online Sales divalidasi berdasarkan beberapa aturan (expectations) seperti keunikan data, kelengkapan nilai, konsistensi tipe data, serta validitas nilaikategorikal.

Hasil validasi ini memastikan bahwa dataset layak digunakan dalam pipeline batch processing sebelum dilakukan transformasi dan analisis lebih lanjut.

## Import Libraries

In [3]:
import pandas as pd
import great_expectations as ge
from great_expectations.data_context import FileDataContext

In [4]:
# Create a data context
context = FileDataContext.create(project_root_dir='./')

## Menghubungkan ke Datasource (Connect to a Datasource)

Pada Great Expectations, perlu mendefinisikan **Datasource** dan **Data Asset** sebelum melakukan validasi kualitas data.

1. Datasource  
Datasource berfungsi sebagai **penghubung antara Great Expectations dengan sumber data**. Datasource menyediakan antarmuka (API) standar untuk mengakses dan berinteraksi dengan data yang berasal dari berbagai sistem, seperti file CSV, database, atau data berbasis Pandas.

Dalam notebook ini, datasource yang digunakan adalah **Pandas Datasource**, karena data dibaca dalam bentuk Pandas DataFrame.

2. Data Asset  
Data asset diberi nama agar mudah dikenali dan dikelola, serta dapat disesuaikan dengan spesifikasi data yang ingin divalidasi.

Sebagai contoh, satu data asset dapat berasal dari satu file CSV atau dari beberapa file yang digabungkan sesuai kebutuhan.
 
Tahap ini bertujuan untuk:
- Mendefinisikan sumber data yang akan digunakan
- Menghubungkan Great Expectations dengan dataset
- Menyiapkan data agar dapat divalidasi pada tahap selanjutnya


In [5]:
# Give a name to a Datasource. This name must be unique between Datasources.
datasource_name = 'Online_Sales'
datasource = context.sources.add_pandas(datasource_name)

# Give a name to a data asset
asset_name = 'Sales'
path_to_data = 'P2M3_Nur_Cahyo_Widodo_data_clean.csv'
asset = datasource.add_csv_asset(asset_name, filepath_or_buffer=path_to_data)

# Build batch request
batch_request = asset.build_batch_request()

Datasource dan data asset berhasil dibuat. Data siap digunakan untuk proses validasi kualitas data menggunakan Great Expectations.

## Membuat Expectation Suite (Create an Expectation Suite)

Expectation Suite merupakan **kumpulan aturan (expectations)** yang digunakan untuk memvalidasi kualitas data. Setiap expectation berisi pernyataan yang dapat diverifikasi terhadap dataset, seperti keunikan nilai, rentang nilai numerik, atau kelengkapan data.

Expectation Suite menggabungkan beberapa expectation menjadi satu deskripsi menyeluruh
mengenai kualitas data.

### Penamaan Expectation Suite
Nama Expectation Suite dapat disesuaikan sesuai kebutuhan, namun **harus bersifat unik**
dalam satu project agar tidak terjadi konflik konfigurasi.

### Validator
Untuk menerapkan expectation ke dataset, digunakan sebuah **Validator**.
Validator berfungsi sebagai penghubung antara:
- Data (batch of data)
- Expectation Suite

Setiap kali sebuah expectation dijalankan menggunakan fungsi `validator.expect_*`,
Great Expectations akan **langsung memvalidasi aturan tersebut terhadap data**.

In [6]:
# Creat an expectation suite
expectation_suite_name = 'expectation-Online-Sales-dataset'
context.add_or_update_expectation_suite(expectation_suite_name)

# Create a validator using above expectation suite
validator = context.get_validator(
    batch_request = batch_request,
    expectation_suite_name = expectation_suite_name
)

# Check the validator
validator.head()

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

,invoiceno,stockcode,description,quantity,invoicedate,unitprice,customerid,country,discount,paymentmethod,shippingcost,category,saleschannel,returnstatus,shipmentprovider,warehouselocation,orderpriority
0,221958,SKU_1964,White Mug,38,2020-01-01 00:00:00,1.71,37039.0,Australia,0.47,Bank Transfer,10.79,Apparel,In-store,Not Returned,UPS,London,Medium
1,771155,SKU_1241,White Mug,18,2020-01-01 01:00:00,41.25,19144.0,Spain,0.19,paypall,9.51,Electronics,Online,Not Returned,UPS,Rome,Medium
2,231932,SKU_1501,Headphones,49,2020-01-01 02:00:00,29.11,50472.0,Germany,0.35,Bank Transfer,23.03,Electronics,Online,Returned,UPS,Berlin,High
3,465838,SKU_1760,Desk Lamp,14,2020-01-01 03:00:00,76.68,96586.0,Netherlands,0.14,paypall,11.08,Accessories,Online,Not Returned,Royal Mail,Rome,Low
4,744167,SKU_1006,Office Chair,47,2020-01-01 05:00:00,70.16,53887.0,Sweden,0.48,Credit Card,13.98,Electronics,Online,Not Returned,DHL,London,Medium


Validator berhasil dibuat dan menampilkan beberapa baris awal dataset. Hal ini menandakan bahwa Expectation Suite telah terhubung dengan data dan siap digunakan untuk proses validasi kualitas data.

### Expectations

In [ ]:
# Expectation 1: Kolom invoiceno harus unik
validator.expect_column_values_to_be_unique("invoiceno")

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": false,
  "result": {
    "element_count": 44804,
    "unexpected_count": 2045,
    "unexpected_percent": 4.564324613873761,
    "partial_unexpected_list": [
      744167,
      427069,
      299041,
      356840,
      168148,
      664685,
      448951,
      589570,
      301664,
      353618,
      835716,
      479989,
      143585,
      125939,
      752664,
      357426,
      781669,
      481050,
      210078,
      411955
    ],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 4.564324613873761,
    "unexpected_percent_nonmissing": 4.564324613873761
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [17]:
# Expectation 2: Kolom quantity harus berada pada rentang nilai yang logis
validator.expect_column_values_to_be_between(
    "quantity",
    min_value=1,
    max_value=1000
)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 44804,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [ ]:
# Expectation 3: Kolom unitprice harus bertipe numerik
validator.expect_column_values_to_be_in_type_list(
    "unitprice",
    ["int", "float"]
)

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "observed_value": "float64"
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [ ]:
# Expectation 4: Kolom country tidak boleh memiliki nilai null
validator.expect_column_values_to_not_be_null("country")

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 44804,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [ ]:
# Expectation 5: Kolom paymentmethod harus sesuai dengan kategori yang ditentukan
validator.expect_column_values_to_match_regex(
    "paymentmethod",
    "^(Credit Card|Bank Transfer|paypall)$"
)


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 44804,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [ ]:
# Expectation 6: Kolom discount harus berada pada rentang 0 sampai 1
validator.expect_column_values_to_be_between(
    "discount",
    min_value=0,
    max_value=1
)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 44804,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [ ]:
# Expectation 7: Kolom saleschannel hanya berisi nilai Online atau In-store
validator.expect_column_values_to_be_in_set(
    "saleschannel",
    ["Online", "In-store"]
)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 44804,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [ ]:
# Save all expectations (both passed and failed) into the expectation suite

validator.save_expectation_suite(discard_failed_expectations=False)

Parameter `discard_failed_expectations=False` digunakan agar seluruh expectation, termasuk yang gagal, tetap tersimpan dalam expectation suite sebagai dokumentasi kualitas data.

In [15]:
# Create a checkpoint

checkpoint_1 = context.add_or_update_checkpoint(
    name = 'checkpoint_1',
    validator = validator,
)

In [16]:
# Run a checkpoint

checkpoint_result = checkpoint_1.run()

Calculating Metrics:   0%|          | 0/43 [00:00<?, ?it/s]